### Set Up Notebook Environment

In [27]:
import sys
import json 

from openai import OpenAI
from openai.types.eval_create_params import DataSourceConfigCustom, TestingCriterionLabelModel


backend_dir = "/Users/eric/Repos/ironbad/backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

In [28]:
from dotenv import load_dotenv
load_dotenv()

True

In [29]:
from app.features.contract_agent.schemas import AgentEvalOutputCase

### Define the Evaluator Input Messages

In [30]:
SYSTEM_MESSAGE = """
You are an expert evaluator tasked with evaluating the peformance of an AI contract review agent. 
Assess the agent's sequence of actions and final output message and determine whether the run is a "pass" or "fail" based on the associated user input message and task success criteria.

# Instructions

1. **Review the User Input Message**: read the user's input message to understand the user's request for this run
2. **Review the Agent Action Sequence**: read the sequence of actions the agent took in response to the user's request. The agent's actions include internal reasoning steps, function calls with the associated function call outputs, and a final response to the user
3. **Review the Agent's Response Message**: at the end of each run the agent responds directly to the user's request and/or summarizes its actions taken during the run
4. **Apply Criteria for Evaluation**: review each task success criterion carefully and determine whether it is satisifed by the agent's sequence of actions and/or final response
5. **Write Step-by-Step Reasoning:**  document each criterion and how the agent's sequence of actions and/or final response meets or fails to meet that criterion. Provide evidence from the sequence of actions and/or final response and explain why it fulfills or does not fulfill the expectations. Provide your reasoning in a bullet-point list
6. **Determine Outcome:** Based on the evaluation, conclude whether the response meets the criteria (Pass) or not (Fail).

# Output Format

Provide the outcome in the following format:

- **Step-by-Step Reasoning:** [Detailed reasoning here]
- **Final Outcome:** "Pass" or "Fail"

# Notes

- If criteria are ambiguous, make a best-effort interpretation and note assumptions made.
"""

In [31]:
USER_MESSAGE = """
# User Input Message
{{item.user_message}}

# Agent Action Sequence
{{item.output_items}}

# Agent Response Message
{{item.assistant_message}}

# Task Success Criteria
{{item.task_criteria}}
"""

### Create the Agent Eval

In [32]:
client = OpenAI()
client

In [ ]:
evals = client.evals.list()
# for eval in evals:
#     client.evals.delete(eval.id)

In [49]:
evals.data

[EvalListResponse(id='eval_69094492fb108191b8d98f83ef41d688', created_at=1762215058, data_source_config=EvalCustomDataSourceConfig(schema_={'required': ['item'], 'properties': {'item': {'required': ['status', 'assistant_message', 'input_items', 'output_items', 'id', 'contract_filename', 'task_category', 'task_criteria', 'user_message'], 'title': 'AgentEvalOutputCase', 'properties': {'contract_filename': {'title': 'Contract Filename', 'type': 'string'}, 'id': {'title': 'Id', 'type': 'integer'}, 'output_items': {'items': {'anyOf': [{'required': ['content', 'role'], 'additionalProperties': True, 'title': 'EasyInputMessage', 'properties': {'content': {'title': 'Content', 'anyOf': [{'type': 'string'}, {'items': {'anyOf': [{'required': ['text', 'type'], 'additionalProperties': True, 'title': 'ResponseInputText', 'properties': {'text': {'title': 'Text', 'type': 'string'}, 'type': {'const': 'input_text', 'title': 'Type', 'type': 'string'}}, 'type': 'object'}, {'required': ['detail', 'type'], '

In [51]:
evals.data[0].nam

EvalListResponse(id='eval_69094492fb108191b8d98f83ef41d688', created_at=1762215058, data_source_config=EvalCustomDataSourceConfig(schema_={'required': ['item'], 'properties': {'item': {'required': ['status', 'assistant_message', 'input_items', 'output_items', 'id', 'contract_filename', 'task_category', 'task_criteria', 'user_message'], 'title': 'AgentEvalOutputCase', 'properties': {'contract_filename': {'title': 'Contract Filename', 'type': 'string'}, 'id': {'title': 'Id', 'type': 'integer'}, 'output_items': {'items': {'anyOf': [{'required': ['content', 'role'], 'additionalProperties': True, 'title': 'EasyInputMessage', 'properties': {'content': {'title': 'Content', 'anyOf': [{'type': 'string'}, {'items': {'anyOf': [{'required': ['text', 'type'], 'additionalProperties': True, 'title': 'ResponseInputText', 'properties': {'text': {'title': 'Text', 'type': 'string'}, 'type': {'const': 'input_text', 'title': 'Type', 'type': 'string'}}, 'type': 'object'}, {'required': ['detail', 'type'], 'a

EvalListResponse(id='eval_69094492fb108191b8d98f83ef41d688', created_at=1762215058, data_source_config=EvalCustomDataSourceConfig(schema_={'required': ['item'], 'properties': {'item': {'required': ['status', 'assistant_message', 'input_items', 'output_items', 'id', 'contract_filename', 'task_category', 'task_criteria', 'user_message'], 'title': 'AgentEvalOutputCase', 'properties': {'contract_filename': {'title': 'Contract Filename', 'type': 'string'}, 'id': {'title': 'Id', 'type': 'integer'}, 'output_items': {'items': {'anyOf': [{'required': ['content', 'role'], 'additionalProperties': True, 'title': 'EasyInputMessage', 'properties': {'content': {'title': 'Content', 'anyOf': [{'type': 'string'}, {'items': {'anyOf': [{'required': ['text', 'type'], 'additionalProperties': True, 'title': 'ResponseInputText', 'properties': {'text': {'title': 'Text', 'type': 'string'}, 'type': {'const': 'input_text', 'title': 'Type', 'type': 'string'}}, 'type': 'object'}, {'required': ['detail', 'type'], 'a

In [39]:
eval = client.evals.create(
    name="Agent Evaluation",
    data_source_config=DataSourceConfigCustom(
        type="custom", 
        item_schema=AgentEvalOutputCase.model_json_schema(),
        include_sample_schema=False
    ),
    testing_criteria=[
        TestingCriterionLabelModel(
            type="label_model",
            name="Task Success",
            model="gpt-5-mini",
            labels=["Pass", "Fail"],
            passing_labels=["Pass"],
            input=[
                {"role": "system", "content": SYSTEM_MESSAGE},
                {"role": "user", "content": USER_MESSAGE}
            ],
        )
    ]
)
eval


EvalCreateResponse(id='eval_69094492fb108191b8d98f83ef41d688', created_at=1762215058, data_source_config=EvalCustomDataSourceConfig(schema_={'type': 'object', 'properties': {'item': {'properties': {'status': {'enum': ['success', 'failure'], 'title': 'Status', 'type': 'string'}, 'assistant_message': {'title': 'Assistant Message', 'type': 'string'}, 'input_items': {'items': {'anyOf': [{'additionalProperties': True, 'properties': {'content': {'anyOf': [{'type': 'string'}, {'items': {'anyOf': [{'additionalProperties': True, 'properties': {'text': {'title': 'Text', 'type': 'string'}, 'type': {'const': 'input_text', 'title': 'Type', 'type': 'string'}}, 'required': ['text', 'type'], 'title': 'ResponseInputText', 'type': 'object'}, {'additionalProperties': True, 'properties': {'detail': {'enum': ['low', 'high', 'auto'], 'title': 'Detail', 'type': 'string'}, 'type': {'const': 'input_image', 'title': 'Type', 'type': 'string'}, 'file_id': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default

### Create an Evaluation Run

In [41]:
data_source = json.load(open("agent_eval_items.json"))
data_source

[{'status': 'success',
  'assistant_message': 'Summary of termination conditions (Section 9)\n\n- Term and renewal: The contract term (and any renewals) is set in the applicable Ordering Document. If the Ordering Document is silent, the Agreement automatically renews annually unless either party gives at least 30 days’ written notice before the end of the then-current term [9].\n\n- Suspension and termination for cause or other reasons: Thomson Reuters may suspend, terminate, or limit Services (or modify terms) on notice if (examples) a third-party provider/court/regulator requests it, you become or are likely to become insolvent, there is or is likely to be a security breach, a breach of your obligations, a breach of TR’s agreement with a third party, a violation of third-party rights, or applicable law. The notice will state the cause and, where remediable, the actions required to reinstate the Service. If you fail to remedy within 30 days (or the cause is not remediable), TR may sus

In [59]:
evals = client.evals.list()
eval = next((eval for eval in evals.data if eval.name == "Agent Evaluation"), None)
eval

EvalListResponse(id='eval_69094492fb108191b8d98f83ef41d688', created_at=1762215058, data_source_config=EvalCustomDataSourceConfig(schema_={'required': ['item'], 'properties': {'item': {'required': ['status', 'assistant_message', 'input_items', 'output_items', 'id', 'contract_filename', 'task_category', 'task_criteria', 'user_message'], 'title': 'AgentEvalOutputCase', 'properties': {'contract_filename': {'title': 'Contract Filename', 'type': 'string'}, 'id': {'title': 'Id', 'type': 'integer'}, 'output_items': {'items': {'anyOf': [{'required': ['content', 'role'], 'additionalProperties': True, 'title': 'EasyInputMessage', 'properties': {'content': {'title': 'Content', 'anyOf': [{'type': 'string'}, {'items': {'anyOf': [{'required': ['text', 'type'], 'additionalProperties': True, 'title': 'ResponseInputText', 'properties': {'text': {'title': 'Text', 'type': 'string'}, 'type': {'const': 'input_text', 'title': 'Type', 'type': 'string'}}, 'type': 'object'}, {'required': ['detail', 'type'], 'a

In [42]:
run = client.evals.runs.create(
    eval_id=eval.id,
    name="Test Run",
    data_source={
        "type": "jsonl",
        "source": {
            "type": "file_content",
            "content": [
                {"item": data_source[0]},
                {"item": data_source[1]},
            ]
        }
    }
)
run
    


RunCreateResponse(id='evalrun_690944f1ee608191bacb0244e9bf8e4d', created_at=1762215153, data_source=CreateEvalJSONLRunDataSource(source=SourceFileContent(content=[SourceFileContentContent(item={'status': 'success', 'assistant_message': 'Summary of termination conditions (Section 9)\n\n- Term and renewal: The contract term (and any renewals) is set in the applicable Ordering Document. If the Ordering Document is silent, the Agreement automatically renews annually unless either party gives at least 30 days’ written notice before the end of the then-current term [9].\n\n- Suspension and termination for cause or other reasons: Thomson Reuters may suspend, terminate, or limit Services (or modify terms) on notice if (examples) a third-party provider/court/regulator requests it, you become or are likely to become insolvent, there is or is likely to be a security breach, a breach of your obligations, a breach of TR’s agreement with a third party, a violation of third-party rights, or applicabl

In [43]:
run.model_dump()

{'id': 'evalrun_690944f1ee608191bacb0244e9bf8e4d',
 'created_at': 1762215153,
 'data_source': {'source': {'content': [{'item': {'status': 'success',
      'assistant_message': 'Summary of termination conditions (Section 9)\n\n- Term and renewal: The contract term (and any renewals) is set in the applicable Ordering Document. If the Ordering Document is silent, the Agreement automatically renews annually unless either party gives at least 30 days’ written notice before the end of the then-current term [9].\n\n- Suspension and termination for cause or other reasons: Thomson Reuters may suspend, terminate, or limit Services (or modify terms) on notice if (examples) a third-party provider/court/regulator requests it, you become or are likely to become insolvent, there is or is likely to be a security breach, a breach of your obligations, a breach of TR’s agreement with a third party, a violation of third-party rights, or applicable law. The notice will state the cause and, where remediable